# Halo Marhaa


# Outfit Recommendation


In [6]:
from langchain_community.tools.tavily_search import TavilySearchResults


from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain.tools import Tool
from dotenv import load_dotenv
import os
import json
import re


load_dotenv()

class FashionSearchAgent:
    def __init__(self):
        # Load environment variables

        self.prompt = """This tool only used for searching product where the input is event or chlotes request. Your task is to search for real, available product urls for fashion items based on the user's event or chlotes request input.

            IMPORTANT SEARCH INSTRUCTIONS:
            1. For each category, first formulate a specific search query like "baju pria formal lebaran" or "sepatu wanita casual"
            2. YOU MUST RETURN DIRECT PRODUCT urls ONLY. Valid urls should point to specific products
            3. Click through to actual product pages to get the direct product URLs
            4. If you cannot find a suitable item after searching, remove that category from the JSON object. Do not include empty categories.
            5. You need to make sure the link is available, because i always found that the link is not available anymore
            6. Only use urls that point directly to specific product pages, never to search results pages.
            7. Use the following categories: "Top", "Bottom", "Outer", "Footwear", "Accessories", "Bag", "Dress", "Jumpsuit", "Swimwear", "Lingerie", "Sleepwear", "Activewear".

            IMPORTANT OUTPUT FORMATTING:
            - Return ONLY the JSON object, without any additional text, explanation, or markdown formatting
            - Do not include code blocks, backticks, or any other non-JSON content

            Generate a JSON object following this exact structure:
            {
                "products": [
                    {
                        "category": "XXX",
                        "name": "<product name>",
                        "urls": [
                            "<direct product url>",
                        ]
                    },
                    {
                        "category": "XXX",
                        "name": "<product name>",
                        "urls": [
                            "<direct product url>",
                        ]
                    }
                ]
            }

            DO NOT MAKE UP urls. Only use urls that point directly to specific product pages, never to search results pages.
            Give me a few of items that match with the event or occasion or the user input.
            """

        # Initialize the LLM
        self.llm = ChatGroq(
            model="llama-3.3-70b-versatile", # ini ganti ke gemini
            temperature=0,
            seed=42,
            top_p=0.002,
            max_retries=15,
        )

        # Create a wrapper function to ensure search results are properly formatted as strings
        def search_and_format(query):
            tavily = TavilySearchResults(
                api_key=os.getenv("TAVILY_API_KEY"),
                max_results=10,
                search_depth='basic',
                include_domains=["shop-id.tokopedia.com"],
                exclude_domains=[
                    "https://www.tokopedia.com/find/",
                    "https://shopee.co.id/search?keyword="
                ]
            )
            results = tavily.invoke(query)
            return results

        self.search_tool = Tool(
            name="product_search_based_on_information",  # Modified name with underscores
            func=search_and_format,
            description=self.prompt,
            max_retries=15,
            verbose=True,  # Set verbose to True for detailed output
            early_stopping_method="generate",  # Added early stopping method
        )
        # Create the agent
        self.agent_executor = create_react_agent(self.llm, [self.search_tool])

    def search(self, user_input):
        response = self.agent_executor.invoke(
            {"messages": user_input},
            {"tool_choice":"required"},
            # Added tool_choice parameter
        )

        content = response['messages'][-1].content
        return content


In [7]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq

from langchain.tools import Tool
from dotenv import load_dotenv
import os
import json
import re

load_dotenv()

class FashionSearchAgentforimage:
    def __init__(self):
        # Load environment variables
        self.prompt = """Tool ini digunakan untuk mencari produk fashion berdasarkan input barang yang diberikan, input akan berupa nama item dan gender. Anda harus mencari produk fashion yang sesuai dengan input yang diberikan.
            Aturan untuk mencari produk:
            1. Lakukan pencarian produk satu per satu berdasarkan input yang diberikan
            2. Gunakan kata kunci yang sesuai dengan kategori produk yang dicari
            3. Pastikan untuk mencari produk yang sesuai dengan kategori yang diberikan

            OUTPUT FORMAT (JSON ONLY, no extra text or markdown):
                        {
                            "products": [
                                {
                                    "category": "XXX",
                                    "name": "<product name>",
                                    "urls": [
                                        "<direct product url>",
                                    ]
                                },
                                {
                                    "category": "XXX",
                                    "name": "<product name>",
                                    "urls": [
                                        "<direct product url>",
                                    ]
                                }
                            ]
                        }

            """
        # Initialize the LLM
        self.llm = ChatGroq(
            model="llama-3.3-70b-versatile", # ini ganti ke gemini
            temperature=0,
            seed=42,
            top_p=0.002,
            max_retries=20,
        )


        # Create a wrapper function to ensure search results are properly formatted as strings
        def search_and_format(query):
            tavily = TavilySearchResults(
                api_key=os.getenv("TAVILY_API_KEY"),
                max_results=10,
                search_depth='basic',
                include_domains=["shop-id.tokopedia.com"],
                exclude_domains=[
                    "https://www.tokopedia.com/find/",
                    "https://shopee.co.id/search?keyword="
                ]
            )
            results = tavily.invoke(query)
            return results

        # Initialize the search tool with the wrapper function
        self.search_tool = Tool(
            name="product_search_based_on_information",  # Modified name with underscores
            func=search_and_format,
            description=self.prompt,
            max_retries=20,
            verbose=True,  # Set verbose to True for detailed output
            early_stopping_method="generate",  # Added early stopping method
        )
        # Create the agent
        self.agent_executor = create_react_agent(self.llm, [self.search_tool])

    def search(self, user_input):
        response = self.agent_executor.invoke(
            {"messages": user_input},
            {"tool_choice":"required"},
            # Added tool_choice parameter
        )

        content = response['messages'][-1].content

        return content

In [8]:
import base64
import os
import requests
from io import BytesIO
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

class OutfitDescriber:
    def __init__(self, model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0, top_p=0.002, seed=42):

        self.client = Groq()
        self.model = model
        self.temperature = temperature
        self.top_p = top_p
        self.seed = seed

    def encode_image(self, image_path):
        # Check if image_path is a URL or a local file path
        if image_path.startswith(('http://', 'https://')):
            # Handle URL
            response = requests.get(image_path)
            if response.status_code == 200:
                return base64.b64encode(response.content).decode('utf-8')
            else:
                raise Exception(f"Failed to fetch image from URL: {response.status_code}")
        else:
            # Handle local file path
            with open(image_path, "rb") as image_file:
                return base64.b64encode(image_file.read()).decode('utf-8')

    def describe_outfit(self, image_path: str) -> str:
        b64_image = self.encode_image(image_path)

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": ("""
                                - Describe the outfit components in the image, and the gender of the person.
                                - Provide the description in a single short paragraph, without using lists or headings.
                                - Include details about: Top, Bottom, Footwear, Head Accessories, Accessories, Bags
                                - If the item is not present, do not include or mention it in the description.
                                - Provide the basic color each item without pattern, and the brand if possible.
                                - Do not talk about missing items or overall color palette.
                                - Do not include any other information or context about the image.
                                - DO NOT PROVIDE A JSON OBJECT OR ANY OTHER FORMATTING.
                                - DO NOT INCLUDE EXPLANATIONS LIKE "She is not wearing" or "He is not wearing.

                                Output format (key dan value hanya ditulis jika ada value, jika tidak ada value tulis none):
                                - gender: <gender>,
                                - Top: <top description>,
                                - Bottom: <bottom description>,
                                - Footwear: <footwear description>,
                                - Head Accessories: <head accessories description>,
                                - Accessories: <accessories description>,
                                - Bags: <bags description>
                                """
                            ),
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{b64_image}",
                            }
                        }
                    ]
                }
            ],
            temperature=self.temperature,
            top_p=self.top_p,
            stream=False,
            seed=self.seed,
        )

        text = response.choices[0].message.content
        # Membagi teks menjadi baris-baris
        lines = text.split('\n')

        # Menyaring baris yang tidak mengandung "none"
        filtered_lines = [line for line in lines if "none" not in line.lower()]

        # Menggabungkan kembali baris-baris yang tersisa
        result = '\n'.join(filtered_lines)
        return result

In [9]:
import json

class FashionAppAgent:
    def __init__(self):
        self.fashion_search_agent = FashionSearchAgent()
        self.outfit_describer = OutfitDescriber()
        self.FashionSearchAgentforimage = FashionSearchAgentforimage()

    def check_input_image_or_event(self, user_input):
        if user_input.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            return "image"
        else:
            return "event"

    def main(self, user_input):
        input_type = self.check_input_image_or_event(user_input)
        outfit_description = None  # Inisialisasi awal
        try:
            if input_type == "image":
                # First get description from image
                outfit_description = self.outfit_describer.describe_outfit(user_input)
                # Then search for products based on that description
                result = self.FashionSearchAgentforimage.search(outfit_description)
            else:
                outfit_description = user_input  # anggap input teks adalah deskripsi
                result = self.fashion_search_agent.search(user_input)
            print(result)
            fixed_data = re.sub(r',(\s*)]', r'\1]', result)
            data_dict = json.loads(fixed_data)
            return data_dict

        except Exception as e:
            raise e

Agent = FashionAppAgent()


c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat_models.py:364: UserWarning: WARNING! seed is not default parameter.
                    seed was transferred to model_kwargs.
                    Please confirm that seed is what you intended.
  warnings.warn(
c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat_models.py:364: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  warnings.warn(
c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat_models.py:364: UserWarning: WARNING! seed is not default parameter.
                    seed was transferred to model_kwargs.
                    Please confirm that seed is what you intended.
  warnings.warn(
c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat

## Usage


In [10]:
# Example usage

# input = r"ootd.jpeg"

input = "male, adult, cafe, casual"
result = Agent.main(input)
print('\n\n',type(result))
print('\n\n', result)

content='[{"title": "Jaket Pria Casualan bomber Eksekutif Man Keren murah vintage - Shop ...", "url": "https://shop-id.tokopedia.com/pdp/1729578039589833600", "content": "Buy Jaket Pria Casualan bomber Eksekutif Man Keren murah vintage on Shop | Tokopedia. Discover great prices on Jaket & Mantel and get free shipping on eligible items. Shop now for exclusive deals!", "score": 0.16297895}, {"title": "CUCI GUDANG Sepatu Pria Casual Sneakers Premium 39-43 TREKKERS MIX", "url": "https://shop-id.tokopedia.com/pdp/1730782004014515315", "content": "Buy CUCI GUDANG Sepatu Pria Casual Sneakers Premium 39-43 TREKKERS on Shop | Tokopedia. Discover great prices on Sepatu Kasual and get free shipping on eligible items. Shop now for exclusive deals!", "score": 0.1262566}, {"title": "Songkok Recca Dewasa - Shop | Tokopedia", "url": "https://shop-id.tokopedia.com/pdp/1729591184464447993", "content": "Buy Songkok Recca Dewasa on Shop | Tokopedia. Discover great prices on Peci and get free shipping on e